# 01. Pollutant Dynamics

This notebook records the first set of Study 3 pollutant assumptions and questions. One step represents one year, and the four regions are the U.S. Census regions. Regional values are scenario inputs, not measurements or calibrated estimates.

The environment accepts parameters, but it is not ready to represent all four pollutants as physical quantities. This notebook lists what still needs to be defined.

## How the current stock equation works

For a pollutant treated as a local mass stock, the scaffold uses:

stock_next = stock_now + emissions_after_prevention + incoming_transport - outgoing_transport - natural_removal - cleanup_mass_removed

Transport should balance across regions unless pollution leaves the modeled area. A transport matrix only makes sense after the modeled areas and pathways are defined.

CO2 uses a different calculation: each emissions pulse contributes to one shared atmospheric stock through the Joos et al. (2013) impulse response:

CO2_perturbation(t) = sum(emission_pulse[k] * R(t - k))

R(age) = 0.2173 + 0.2240 exp(-age / 394.4) + 0.2824 exp(-age / 36.54) + 0.2763 exp(-age / 4.304)

Age is in years. The code treats the starting CO2 value as a new pulse. This is a temporary convention, not an observed baseline or historical profile. [Joos et al.](https://acp.copernicus.org/articles/13/2793/2013/)

The local-stock equation does not fit every pollutant. PM2.5 needs an annual exposure calculation. Nitrate concentration depends on water mass and volume. Lead immobilization changes mobility but does not remove total lead.

## What is still needed

| Pollutant | What sources tell us | What I still need to define |
|---|---|---|
| PM2.5 | Annual exposure is measured as air concentration; transport and removal depend on atmospheric conditions. | Emissions, source-to-exposure mapping, monitor aggregation, and controls. |
| CO2 | The atmosphere is shared; removal happens over multiple time scales. | Emissions boundary, units, starting baseline, capture, and removal. |
| Nitrate-N | Nitrate moves through connected water systems. | Catchment, water volume, flow, nitrogen inputs, transformations, and treatment. |
| Lead in soil | Lead persists; mobility depends on soil conditions. | Total and mobile quantities, soil compartment, inputs, transport, and remediation. |

The [source and parameter specification](../source_and_parameter_specification.md) lists the sources and open questions.

In [ ]:
import pandas as pd

region_parameters = pd.DataFrame({
    "region": ["Northeast", "Midwest", "South", "West"],
    "initial_edi": [70, 60, 55, 75],
    "initial_currency": [700, 550, 450, 800],
    "annual_income": [65, 55, 50, 75],
    "economic_growth": [0.015, 0.012, 0.020, 0.018],
    "pm25_multiplier": [0.9, 1.2, 1.1, 0.9],
    "co2_multiplier": [0.9, 1.2, 1.1, 1.0],
    "nitrate_multiplier": [0.8, 1.4, 1.2, 0.8],
    "lead_multiplier": [0.9, 1.3, 1.0, 0.8],
})

display(region_parameters)

In [ ]:
pollutant_parameters = pd.DataFrame({
    "pollutant": ["PM2.5", "CO2", "Nitrate", "Lead"],
    "initial_index": [15, 20, 15, 10],
    "annual_emission_input": [12, 10, 8, 4],
    "annual_retention": [0.0, None, 0.80, 1.0],
    "spillover": [0.20, None, 0.10, 0.01],
    "prevention_efficiency": [0.70, 0.60, 0.60, 0.50],
    "cleanup_delay_years": [1, None, 2, 3],
    "status": ["Assumed"] * 4,
})
display(pollutant_parameters)

In [ ]:
parameter_status = pd.DataFrame([
    {
        "pollutant": "PM2.5",
        "stock_unit": "to define",
        "annual_emissions_source": "EPA AP-42 or source-specific inventory",
        "transport_and_removal": "requires airshed scope and parameter evidence",
        "status": "not calibrated",
    },
    {
        "pollutant": "CO2",
        "stock_unit": "to define",
        "annual_emissions_source": "EPA GHG factors or source-specific inventory",
        "transport_and_removal": "shared global stock; sink model to specify",
        "status": "not calibrated",
    },
    {
        "pollutant": "Nitrate-N",
        "stock_unit": "water volume and concentration basis to define",
        "annual_emissions_source": "USGS or watershed-specific annual load",
        "transport_and_removal": "requires water compartment and flow data",
        "status": "not calibrated",
    },
    {
        "pollutant": "Lead (Pb)",
        "stock_unit": "soil concentration and mobile fraction to define",
        "annual_emissions_source": "source-specific inventory or measured load",
        "transport_and_removal": "persistent stock; distinguish removal from immobilization",
        "status": "soil compartment selected; units not calibrated",
    },
])

display(parameter_status)

## Reading the proposed regional values

The regional and pollutant values came from the initial Study 3 proposal. They are assumptions, not measurements. The proposed emissions calculation is baseline input times regional multiplier. It produces scenario units because the baseline inputs have no physical units yet.

For example, the proposed Northeast PM2.5 input is 12 times 0.9, or 10.8 scenario units per year. The proposed Midwest nitrate input is 8 times 1.4, or 11.2 scenario units per year. Neither value is a measured load.

The CO2 code applies the Joos response to emissions cohorts. Its starting index is treated as an age-zero pulse because no historical profile has been selected. PM2.5 still needs a separate equation that maps emissions to annual exposure.

In [ ]:
pollutant_baselines = {"PM2.5": 12, "CO2": 10, "Nitrate-N": 8, "Lead": 4}

regional_multiplier_columns = {"PM2.5": "pm25_multiplier", "CO2": "co2_multiplier", "Nitrate-N": "nitrate_multiplier",
                               "Lead": "lead_multiplier"}

regional_emission_inputs = pd.DataFrame({"region": region_parameters["region"]})

for pollutant, multiplier_column in regional_multiplier_columns.items():
    regional_emission_inputs[pollutant] = (pollutant_baselines[pollutant] * region_parameters[multiplier_column])

display(regional_emission_inputs.round(2))

## Retention and thresholds

The proposed retention values cannot be used the same way for every pollutant. PM2.5 is an annual average exposure, not a stock with a simple yearly retention rate. CO2 is a shared stock with a multi-timescale response. Nitrate retention only describes a simplified mass balance after defining the water compartment and flows. Lead does not decay as an element, so total lead should be separated from its mobile fraction.

Published values provide context, but they are not interchangeable failure thresholds: EPA's annual PM2.5 standard is 9 micrograms per cubic meter averaged over three years; the U.S. drinking-water nitrate limit is 10 mg/L as nitrogen; EPA's residential soil-lead screening level is 200 ppm for a specific cleanup context. [EPA PM2.5 standard](https://www.epa.gov/system/files/documents/2024-04/2024-pm-naaqs-fr-published.pdf) | [EPA nitrate limit](https://nepis.epa.gov/Exe/ZyURL.cgi?Dockey=P10150PM.TXT) | [EPA lead directive](https://www.epa.gov/superfund/residential-soil-lead-directive-cercla-sites-and-rcra-hazardous-waste-cleanup-facilities)